# DreamVision 2.0: Background GAN with Places365

This notebook trains a background-only DCGAN on Places365 scene images.

Goal for phase 1:
- learn strong background composition from a large scene dataset
- avoid mixing character/action generation into the same model
- save realistic background generations for later compositing and cartoon post-processing


## HiPerGator Notes

- Target runtime: PyTorch 2.7.0
- Recommended: GPU notebook session
- Put Places365 under `DreamVision 2.0/data/places365/`
- The code below assumes an ImageFolder-style layout with class folders under `train/` and optionally `val/`


In [ ]:
from __future__ import annotations

import math
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import ImageFile
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [ ]:
@dataclass
class BackgroundGANConfig:
    project_root: Path = Path.cwd().resolve().parent
    data_root: Path = project_root / "data" / "places365"
    train_dir: Path = data_root / "train"
    val_dir: Path = data_root / "val"
    output_root: Path = project_root / "outputs" / "backgrounds"
    checkpoint_dir: Path = output_root / "checkpoints"
    sample_dir: Path = output_root / "samples"
    image_size: int = 128
    batch_size: int = 128
    num_workers: int = 8
    latent_dim: int = 128
    ngf: int = 64
    ndf: int = 64
    num_channels: int = 3
    num_epochs: int = 50
    learning_rate: float = 2e-4
    beta1: float = 0.5
    sample_interval: int = 1
    checkpoint_interval: int = 5
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


config = BackgroundGANConfig()
config.checkpoint_dir.mkdir(parents=True, exist_ok=True)
config.sample_dir.mkdir(parents=True, exist_ok=True)

print(asdict(config))


In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


set_seed(config.seed)
device = torch.device(config.device)
device


## Dataset Loader

This first notebook ignores the Places365 labels during training. We only use the scene images to learn general background structure.

Later, once the background GAN is stable, we can decide whether to:
- keep it unconditional for broad scene variety
- or make it conditional on a reduced scene taxonomy like `forest`, `bedroom`, `street`, `park`, and `store`


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(int(config.image_size * 1.1)),
    transforms.CenterCrop(config.image_size),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

if not config.train_dir.exists():
    raise FileNotFoundError(
        f"Places365 train directory not found: {config.train_dir}\n"
        "Download or copy the dataset to the expected location before training."
    )

train_dataset = datasets.ImageFolder(root=str(config.train_dir), transform=train_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
    persistent_workers=config.num_workers > 0,
)

print(f"Training images: {len(train_dataset):,}")
print(f"Scene classes discovered: {len(train_dataset.classes):,}")
print(f"Batches per epoch: {len(train_loader):,}")


In [ ]:
def denormalize(images: torch.Tensor) -> torch.Tensor:
    return (images * 0.5 + 0.5).clamp(0, 1)


batch_images, batch_labels = next(iter(train_loader))
grid = utils.make_grid(denormalize(batch_images[:16]), nrow=4)

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.imshow(grid.permute(1, 2, 0))
plt.show()


## DCGAN Model

This is intentionally simple for the first DreamVision 2.0 pass. If training is stable and outputs look promising, we can later upgrade to a stronger backbone or a conditional background model.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim: int, ngf: int, num_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 16),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 16, ngf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, num_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, noise: torch.Tensor) -> torch.Tensor:
        return self.net(noise)


class Discriminator(nn.Module):
    def __init__(self, ndf: int, num_channels: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(num_channels, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, ndf * 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 16),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 16, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.net(images).view(-1)


def weights_init(module: nn.Module) -> None:
    classname = module.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(module.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(module.weight.data, 1.0, 0.02)
        nn.init.constant_(module.bias.data, 0)


In [ ]:
generator = Generator(config.latent_dim, config.ngf, config.num_channels).to(device)
discriminator = Discriminator(config.ndf, config.num_channels).to(device)

generator.apply(weights_init)
discriminator.apply(weights_init)

criterion = nn.BCELoss()
optimizer_g = optim.Adam(generator.parameters(), lr=config.learning_rate, betas=(config.beta1, 0.999))
optimizer_d = optim.Adam(discriminator.parameters(), lr=config.learning_rate, betas=(config.beta1, 0.999))

fixed_noise = torch.randn(16, config.latent_dim, 1, 1, device=device)

sum(p.numel() for p in generator.parameters()), sum(p.numel() for p in discriminator.parameters())


In [ ]:
def save_checkpoint(epoch: int) -> None:
    torch.save(
        {
            "epoch": epoch,
            "config": asdict(config),
            "generator_state_dict": generator.state_dict(),
            "discriminator_state_dict": discriminator.state_dict(),
            "optimizer_g_state_dict": optimizer_g.state_dict(),
            "optimizer_d_state_dict": optimizer_d.state_dict(),
        },
        config.checkpoint_dir / f"background_gan_epoch_{epoch:03d}.pt",
    )


def save_sample_grid(epoch: int) -> Path:
    generator.eval()
    with torch.no_grad():
        samples = generator(fixed_noise).cpu()
    generator.train()

    sample_path = config.sample_dir / f"background_epoch_{epoch:03d}.png"
    utils.save_image(denormalize(samples), sample_path, nrow=4)
    return sample_path


In [ ]:
history = []

for epoch in range(1, config.num_epochs + 1):
    start_time = time.time()
    g_running = 0.0
    d_running = 0.0

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{config.num_epochs}")
    for real_images, _ in progress:
        real_images = real_images.to(device, non_blocking=True)
        batch_size = real_images.size(0)

        real_targets = torch.ones(batch_size, device=device)
        fake_targets = torch.zeros(batch_size, device=device)

        discriminator.zero_grad(set_to_none=True)

        real_output = discriminator(real_images)
        d_loss_real = criterion(real_output, real_targets)

        noise = torch.randn(batch_size, config.latent_dim, 1, 1, device=device)
        fake_images = generator(noise)
        fake_output = discriminator(fake_images.detach())
        d_loss_fake = criterion(fake_output, fake_targets)

        d_loss = d_loss_real + d_loss_fake
        d_loss.backward()
        optimizer_d.step()

        generator.zero_grad(set_to_none=True)

        gen_output = discriminator(fake_images)
        g_loss = criterion(gen_output, real_targets)
        g_loss.backward()
        optimizer_g.step()

        g_running += g_loss.item()
        d_running += d_loss.item()
        progress.set_postfix(d_loss=f"{d_loss.item():.4f}", g_loss=f"{g_loss.item():.4f}")

    avg_g = g_running / len(train_loader)
    avg_d = d_running / len(train_loader)
    elapsed = time.time() - start_time

    history.append({
        "epoch": epoch,
        "g_loss": avg_g,
        "d_loss": avg_d,
        "epoch_time_sec": elapsed,
    })

    print(f"Epoch {epoch:03d} | D: {avg_d:.4f} | G: {avg_g:.4f} | time: {elapsed:.1f}s")

    if epoch % config.sample_interval == 0:
        sample_path = save_sample_grid(epoch)
        print(f"Saved samples to {sample_path}")

    if epoch % config.checkpoint_interval == 0:
        save_checkpoint(epoch)
        print(f"Saved checkpoint for epoch {epoch}")


In [ ]:
epochs = [row["epoch"] for row in history]
g_losses = [row["g_loss"] for row in history]
d_losses = [row["d_loss"] for row in history]

plt.figure(figsize=(10, 5))
plt.plot(epochs, g_losses, label="Generator")
plt.plot(epochs, d_losses, label="Discriminator")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Background GAN Training Loss")
plt.legend()
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
generator.eval()
with torch.no_grad():
    preview_noise = torch.randn(16, config.latent_dim, 1, 1, device=device)
    preview_images = generator(preview_noise).cpu()

preview_grid = utils.make_grid(denormalize(preview_images), nrow=4)
plt.figure(figsize=(8, 8))
plt.axis("off")
plt.imshow(preview_grid.permute(1, 2, 0))
plt.show()


## Next Steps

After we validate this notebook on HiPerGator, the likely follow-ups are:
- improve output resolution or training stability
- restrict Places365 to a curated subset of story-friendly scene classes
- train a separate character/action model
- composite subject over generated background
- apply a cartoon or animation-style filter to unify the final image
